In [ ]:
import tkinter as tk
from tkinter import ttk, scrolledtext, messagebox
import joblib
import re
import string
import csv
import os
from datetime import datetime

model = joblib.load("best_lightgbm_model.pkl")
tfidf = joblib.load("tfidf_vectorizer.pkl")


def clean_text(text):
    text = text.lower()
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'\S+@\S+', '', text)
    text = re.sub(r'\d+', '', text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def predict():
    combined = (
        title_entry.get() + " " +
        company_box.get("1.0", tk.END) + " " +
        description_box.get("1.0", tk.END) + " " +
        requirements_box.get("1.0", tk.END) + " " +
        benefits_box.get("1.0", tk.END)
    )

    if combined.strip() == "":
        messagebox.showwarning(
            "Warning",
            "Please enter job details."
        )
        return

    cleaned = clean_text(combined)

    vec = tfidf.transform([cleaned])

    pred = model.predict(vec)[0]
    prob = model.predict_proba(vec)[0]

    conf = prob.max() * 100

    if pred == 1:
        prediction_var.set("FAKE JOB POSTING")
        prediction_label.config(fg="red")
    else:
        prediction_var.set("REAL JOB POSTING")
        prediction_label.config(fg="green")

    confidence_var.set(f"{conf:.2f}%")
    real_var.set(f"{prob[0] * 100:.2f}%")
    fake_var.set(f"{prob[1] * 100:.2f}%")


def reset():
    title_entry.delete(0, tk.END)

    for w in [
        company_box,
        description_box,
        requirements_box,
        benefits_box,
        comments_box
    ]:
        w.delete("1.0", tk.END)

    prediction_var.set("")
    confidence_var.set("")
    real_var.set("")
    fake_var.set("")

    prediction_label.config(fg="black")

    rating.set("Excellent")


def save_feedback():
    if prediction_var.get().strip() == "":
        messagebox.showwarning(
            "Warning",
            "Please make a prediction before submitting feedback."
        )
        return

    file = "prototype_feedback.csv"

    exists = os.path.exists(file)

    with open(
        file,
        "a",
        newline="",
        encoding="utf-8"
    ) as f:

        writer = csv.writer(f)

        if not exists:
            writer.writerow([
                "Date",
                "Prediction",
                "Confidence",
                "Rating",
                "Comments"
            ])

        writer.writerow([
            datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            prediction_var.get(),
            confidence_var.get(),
            rating.get(),
            comments_box.get("1.0", tk.END).strip()
        ])

    messagebox.showinfo(
        "Saved",
        "Feedback saved successfully."
    )


root = tk.Tk()

root.title("Fake Job Posting Detection Prototype")
root.geometry("1100x850")
root.minsize(800, 600)
root.configure(bg="#eaf4ff")

style = ttk.Style()
style.theme_use("clam")

title = tk.Label(
    root,
    text="Fake Job Posting Detection Using Machine Learning",
    bg="#1565C0",
    fg="white",
    font=("Arial", 22, "bold"),
    pady=12
)

title.pack(fill="x")

container = tk.Frame(
    root,
    bg="#eaf4ff"
)

container.pack(
    fill="both",
    expand=True
)

canvas = tk.Canvas(
    container,
    bg="#eaf4ff",
    highlightthickness=0
)

canvas.pack(
    side="left",
    fill="both",
    expand=True
)

scrollbar = ttk.Scrollbar(
    container,
    orient="vertical",
    command=canvas.yview
)

scrollbar.pack(
    side="right",
    fill="y"
)

canvas.configure(
    yscrollcommand=scrollbar.set
)

main = tk.Frame(
    canvas,
    bg="#eaf4ff"
)

canvas_window = canvas.create_window(
    (0, 0),
    window=main,
    anchor="nw"
)


def update_scroll_region(event=None):
    canvas.configure(
        scrollregion=canvas.bbox("all")
    )


main.bind(
    "<Configure>",
    update_scroll_region
)


def resize_main(event):
    canvas.itemconfig(
        canvas_window,
        width=event.width
    )


canvas.bind(
    "<Configure>",
    resize_main
)


def mouse_wheel(event):
    canvas.yview_scroll(
        int(-1 * (event.delta / 120)),
        "units"
    )


canvas.bind_all(
    "<MouseWheel>",
    mouse_wheel
)


def add_box(label, height=3):
    tk.Label(
        main,
        text=label,
        font=("Arial", 11, "bold"),
        bg="#eaf4ff"
    ).pack(
        anchor="w",
        padx=20,
        pady=(8, 2)
    )

    box = scrolledtext.ScrolledText(
        main,
        width=120,
        height=height,
        font=("Arial", 10),
        wrap=tk.WORD
    )

    box.pack(
        fill="x",
        padx=20,
        pady=4
    )

    return box


tk.Label(
    main,
    text="Job Title",
    font=("Arial", 11, "bold"),
    bg="#eaf4ff"
).pack(
    anchor="w",
    padx=20,
    pady=(10, 2)
)

title_entry = tk.Entry(
    main,
    font=("Arial", 11)
)

title_entry.pack(
    fill="x",
    padx=20,
    pady=4
)

company_box = add_box(
    "Company Profile",
    4
)

description_box = add_box(
    "Job Description",
    8
)

requirements_box = add_box(
    "Requirements",
    4
)

benefits_box = add_box(
    "Benefits",
    4
)

btn = tk.Frame(
    main,
    bg="#eaf4ff"
)

btn.pack(
    pady=15
)

predict_button = tk.Button(
    btn,
    text="Predict",
    bg="#1976D2",
    fg="white",
    font=("Arial", 11, "bold"),
    width=15,
    command=predict
)

predict_button.grid(
    row=0,
    column=0,
    padx=8
)

reset_button = tk.Button(
    btn,
    text="Reset",
    bg="#F9A825",
    fg="white",
    font=("Arial", 11, "bold"),
    width=15,
    command=reset
)

reset_button.grid(
    row=0,
    column=1,
    padx=8
)

exit_button = tk.Button(
    btn,
    text="Exit",
    bg="#D32F2F",
    fg="white",
    font=("Arial", 11, "bold"),
    width=15,
    command=root.destroy
)

exit_button.grid(
    row=0,
    column=2,
    padx=8
)

result = tk.LabelFrame(
    main,
    text="Prediction Result",
    font=("Arial", 12, "bold"),
    bg="white",
    padx=10,
    pady=10
)

result.pack(
    fill="x",
    padx=20,
    pady=10
)

prediction_var = tk.StringVar()
confidence_var = tk.StringVar()
real_var = tk.StringVar()
fake_var = tk.StringVar()

prediction_label = tk.Label(
    result,
    textvariable=prediction_var,
    bg="white",
    font=("Arial", 18, "bold")
)

prediction_label.pack(
    pady=8
)

for text, var in [
    ("Confidence", confidence_var),
    ("Real Probability", real_var),
    ("Fake Probability", fake_var)
]:

    tk.Label(
        result,
        text=text + ":",
        bg="white",
        font=("Arial", 11, "bold")
    ).pack(
        pady=(3, 0)
    )

    tk.Label(
        result,
        textvariable=var,
        bg="white",
        font=("Arial", 11)
    ).pack(
        pady=(0, 3)
    )

feedback = tk.LabelFrame(
    main,
    text="Human Evaluation Feedback",
    font=("Arial", 12, "bold"),
    bg="white",
    padx=10,
    pady=10
)

feedback.pack(
    fill="x",
    padx=20,
    pady=10
)

rating = tk.StringVar(
    value="Excellent"
)

for value in [
    "Excellent",
    "Good",
    "Average",
    "Poor"
]:

    tk.Radiobutton(
        feedback,
        text=value,
        variable=rating,
        value=value,
        bg="white",
        font=("Arial", 10)
    ).pack(
        anchor="w",
        padx=10,
        pady=1
    )

tk.Label(
    feedback,
    text="Comments",
    bg="white",
    font=("Arial", 11, "bold")
).pack(
    anchor="w",
    padx=10,
    pady=(8, 3)
)

comments_box = scrolledtext.ScrolledText(
    feedback,
    height=5,
    font=("Arial", 10),
    wrap=tk.WORD
)

comments_box.pack(
    fill="x",
    padx=10,
    pady=5
)

save_button = tk.Button(
    feedback,
    text="Save Feedback",
    bg="#388E3C",
    fg="white",
    font=("Arial", 11, "bold"),
    width=20,
    command=save_feedback
)

save_button.pack(
    pady=10
)

root.mainloop()